In [1]:
import xgboost as xgb

print(xgb.__version__)

3.0.4


In [2]:
import sys

import xgboost as xgb

print(sys.executable)        # should point to .../.venv/bin/python
print(xgb.__version__)       # should print 3.0.4
print(xgb.__file__)          # should live under .../.venv/...

/Users/manthanvyas/Desktop/housing-price-prediction-mlops/.venv/bin/python
3.0.4
/Users/manthanvyas/Desktop/housing-price-prediction-mlops/.venv/lib/python3.11/site-packages/xgboost/__init__.py


In [3]:
# ==============================================
# 1. Imports
# ==============================================
import mlflow
import mlflow.xgboost
import numpy as np
import optuna
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor

/Users/manthanvyas/Desktop/housing-price-prediction-mlops/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# ==============================================
# 2. Load processed datasets
# ==============================================
train_df = pd.read_csv("/Users/manthanvyas/Desktop/housing-price-prediction-mlops/data/processed/feature_engineered_train.csv")
eval_df  = pd.read_csv("/Users/manthanvyas/Desktop/housing-price-prediction-mlops/data/processed/feature_engineered_eval.csv")

# Define target + features
target = "price"
X_train, y_train = train_df.drop(columns = [target]), train_df[target]
X_eval, y_eval   = eval_df.drop(columns = [target]), eval_df[target]

print("Train shape:", X_train.shape)
print("Eval shape:", X_eval.shape)

Train shape: (576815, 39)
Eval shape: (148448, 39)


In [5]:
# ==============================================
# 3. Define Optuna objective function with MLflow
# ==============================================
def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 200, 1000),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log = True),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "gamma": trial.suggest_float("gamma", 0.0, 5.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log = True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log = True),
        "random_state": 42,
        "n_jobs": -1,
        "tree_method": "hist",
    }

    with mlflow.start_run(nested = True):
        model = XGBRegressor(**params)
        model.fit(X_train, y_train)

        y_pred = model.predict(X_eval)
        rmse = float(np.sqrt(mean_squared_error(y_eval, y_pred)))
        mae = float(mean_absolute_error(y_eval, y_pred))
        r2 = float(r2_score(y_eval, y_pred))

        # Log hyperparameters + metrics
        mlflow.log_params(params)
        mlflow.log_metrics({"rmse": rmse, "mae": mae, "r2": r2})

    return rmse

In [6]:
# ==============================================
# 4. Run Optuna study with MLflow
# ==============================================
# Force MLflow to always use the root project mlruns folder

mlflow.set_tracking_uri("/Users/manthanvyas/Desktop/housing-price-prediction-mlops/mlruns")
mlflow.set_experiment("xgboost_optuna_housing")

study = optuna.create_study(direction = "minimize")
study.optimize(objective, n_trials = 15)

print("Best params:", study.best_trial.params)

[I 2026-07-10 20:24:59,733] A new study created in memory with name: no-name-a5e15581-bc1d-4606-a57e-e0503e19df3b
[I 2026-07-10 20:25:10,276] Trial 0 finished with value: 78346.88007976297 and parameters: {'n_estimators': 839, 'max_depth': 6, 'learning_rate': 0.016848128779152646, 'subsample': 0.9883928915390898, 'colsample_bytree': 0.949274601200153, 'min_child_weight': 3, 'gamma': 0.014998254235839004, 'reg_alpha': 4.5340091737482934e-08, 'reg_lambda': 0.05500266616088646}. Best is trial 0 with value: 78346.88007976297.
[I 2026-07-10 20:25:22,487] Trial 1 finished with value: 70520.23669083533 and parameters: {'n_estimators': 446, 'max_depth': 10, 'learning_rate': 0.12372747700196125, 'subsample': 0.9212547047211026, 'colsample_bytree': 0.6446944788284088, 'min_child_weight': 8, 'gamma': 2.8898715490089493, 'reg_alpha': 0.03165729696842299, 'reg_lambda': 8.684885777084462}. Best is trial 1 with value: 70520.23669083533.
[I 2026-07-10 20:25:35,941] Trial 2 finished with value: 79494.9

Best params: {'n_estimators': 703, 'max_depth': 7, 'learning_rate': 0.04309908505820415, 'subsample': 0.6510028924383973, 'colsample_bytree': 0.5070804549185479, 'min_child_weight': 7, 'gamma': 1.6037796319858908, 'reg_alpha': 9.926743176674815, 'reg_lambda': 0.0003838389326013394}


In [7]:
# ==============================================
# 5. Train final model with best params and log to MLflow
# ==============================================
best_params = study.best_trial.params
best_model = XGBRegressor(**best_params)
best_model.fit(X_train, y_train)

y_pred = best_model.predict(X_eval)

mae = mean_absolute_error(y_eval, y_pred)
rmse = np.sqrt(mean_squared_error(y_eval, y_pred))
r2 = r2_score(y_eval, y_pred)

print("Final tuned model performance:")
print("MAE:", mae)
print("RMSE:", rmse)
print("R²:", r2)

# Log final model
with mlflow.start_run(run_name = "best_xgboost_model"):
    mlflow.log_params(best_params)
    mlflow.log_metrics({"rmse": rmse, "mae": mae, "r2": r2})
    best_model._estimator_type = "regressor" # type: ignore
    mlflow.xgboost.log_model(best_model, name = "model") # type: ignore

Final tuned model performance:
MAE: 31938.29449287179
RMSE: 68997.7162949737
R²: 0.9632100274037747


/Users/manthanvyas/Desktop/housing-price-prediction-mlops/.venv/lib/python3.11/site-packages/xgboost/sklearn.py:1028: UserWarning: [20:28:05] WARNING: /Users/runner/work/xgboost/xgboost/src/c_api/c_api.cc:1427: Saving model in the UBJSON format as default.  You can use file extension: `json`, `ubj` or `deprecated` to choose between formats.
  self.get_booster().save_model(fname)
2026/07/10 20:28:08 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: /var/folders/n_/jw4v2htd70vgxxhm11wf4plh0000gn/T/tmpjbcxc140/model, flavor: xgboost). Fall back to return ['xgboost==3.0.4']. Set logging level to DEBUG to see the full traceback. 
2026/07/10 20:28:08 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
2026/07/10 20:28:08 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` p